# Tree Data Extraction and Analysis

This notebook demonstrates how to load, inspect, and extract tree positions and heights from the local GeoPackage (`.gpkg`) tree database (`Trees/125015_baeume.gpkg`).

In [1]:
import os
import pandas as pd
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
from shapely.geometry import box
from pyproj import Transformer

### 2. Path

In [2]:
TREE_GPKG_PATH = '../Trees/125015_baeume.gpkg'


### 3. Load the Tree Dataset

In [3]:
print("Loading tree dataset (this may take 10-15 seconds)...")
trees_gdf = gpd.read_file(TREE_GPKG_PATH)
print(f"Loaded {len(trees_gdf):,} trees.")
print("CRS of the dataset:", trees_gdf.crs)
trees_gdf.head()

Loading tree dataset (this may take 10-15 seconds)...
Loaded 3,789,054 trees.
CRS of the dataset: EPSG:25832


,dgmhoehe,baumhoehe,geometry
0,260.7,28.9,POINT (562456.1 5531982.3)
1,259.5,28.4,POINT (562470.5 5531994.5)
2,269.2,28.3,POINT (562509.1 5531956.1)
3,258.7,28.1,POINT (562465.5 5531997.1)
4,265.4,28.1,POINT (562487.5 5531967.5)


### 4. Extract Tree Features

In [4]:
wuerzburg_trees = trees_gdf
wuerzburg_trees['Tree_Position_X'] = wuerzburg_trees.geometry.x
wuerzburg_trees['Tree_Position_Y'] = wuerzburg_trees.geometry.y

columns_to_keep = ['Tree_Position_X', 'Tree_Position_Y', 'baumhoehe', 'dgmhoehe']
optional_keywords = ['crown', 'kronen', 'diameter', 'radius', 'shape', 'breite', 'durchmesser']
for col in wuerzburg_trees.columns:
    if col.lower() != 'geometry' and col not in columns_to_keep:
        if any(keyword in col.lower() for keyword in optional_keywords):
            columns_to_keep.append(col)

wuerzburg_trees = wuerzburg_trees[columns_to_keep]
wuerzburg_trees.head()

,Tree_Position_X,Tree_Position_Y,baumhoehe,dgmhoehe
0,562456.1,5531982.3,28.9,260.7
1,562470.5,5531994.5,28.4,259.5
2,562509.1,5531956.1,28.3,269.2
3,562465.5,5531997.1,28.1,258.7
4,562487.5,5531967.5,28.1,265.4


### 5. Inspect Tree Heights and Statistics

In [5]:
print("Tree Height (baumhoehe) statistics (meters):")
print(wuerzburg_trees['baumhoehe'].describe())

print("\nGround Elevation (dgmhoehe) statistics (meters):")
print(wuerzburg_trees['dgmhoehe'].describe())

Tree Height (baumhoehe) statistics (meters):
count    3.789054e+06
mean     2.088497e+01
std      7.748716e+00
min      5.500000e+00
25%      1.510000e+01
50%      2.200000e+01
75%      2.670000e+01
max      6.480000e+01
Name: baumhoehe, dtype: float64

Ground Elevation (dgmhoehe) statistics (meters):
count    3.789054e+06
mean     2.987430e+02
std      6.955695e+01
min      1.648000e+02
25%      2.453000e+02
50%      2.932000e+02
75%      3.407000e+02
max      4.779000e+02
Name: dgmhoehe, dtype: float64


### 6. Output

In [6]:
import os
import json
out_dir = "outputs"
if not os.path.exists(out_dir) and os.path.exists("../outputs"):
    out_dir = "../outputs"

# Save trees CSV
wuerzburg_trees.to_csv(os.path.join(out_dir, "wuerzburg_extracted_trees.csv"), index=False)

# Save trees JSON cleanly by converting to records first
wuerzburg_trees_dict = wuerzburg_trees.to_dict(orient="records")
with open(os.path.join(out_dir, "wuerzburg_extracted_trees.json"), "w", encoding="utf-8") as f:
    json.dump(wuerzburg_trees_dict, f)
